# Rebin BSM signal shapes with observed histogram binning

In [1]:
# In case of problems with importing libraries
# Go to "Softwares" (blue cube symbol on the left)
# Load the python/3.9.6 ipython-kernel/3.9 modules
# Load scipy-stack/2023a StdEnv/2020 gcc/9.3.0 root/6.26.06
# then select the Python 3.9 kernel on the top right
import numpy as np
from pathlib import Path
from multiprocessing import Pool, cpu_count
from os import listdir
from tqdm import tqdm

import pandas as pd
from copy import deepcopy
import ROOT

Welcome to JupyROOT 6.26/06


## Load functions

In [2]:
def rebin(hist, binning, norm_by_width=False):
    """ Rebins histogram based on provided binning.

    Arguments:
        binning (``list``): list of bin edges
        norm_by_width (``bool``): whether to normalize by bin width
    """
    from array import array
    from math import sqrt
    
    name = hist.GetTitle()

    # # Supress warning for replacing histogram
    # ignore_level = ROOT.gErrorIgnoreLevel
    # ROOT.gErrorIgnoreLevel = ROOT.kError
    
    reb_hist = ROOT.TH1D(name, name, len(binning)-1, array('d', binning))   
       
    # ROOT.gErrorIgnoreLevel = ignore_level
    reb_hist.Sumw2()

    nbinsx = [ i for i in range(hist.GetNbinsX()+2) ]
    for binx in nbinsx:
        old_center_x = hist.GetXaxis().GetBinCenter(binx)
        new_global_bin = reb_hist.FindBin(old_center_x)
        reb_hist.SetBinContent(new_global_bin,hist.GetBinContent(binx)+reb_hist.GetBinContent(new_global_bin))
        error = hist.GetBinError(binx)**2+reb_hist.GetBinError(new_global_bin)**2
        reb_hist.SetBinError(new_global_bin,sqrt(error))
                       
    if norm_by_width:
        reb_hist.Scale(1, "width")

    return reb_hist

## Config

In [3]:
# Define configuration variables
observed_dir = '/project/def-arguinj/shared/DDP_data/2024-01-19_dark_machines_eva_analysis12/v10_3'
signal_dir = '/project/def-arguinj/shared/DDP_data/2023-11-08_dark_machines_samuel_analysis10_bsm_signals'
output_dir = '/project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3'
Path(output_dir).mkdir(parents=True, exist_ok=True)
file_name = 'histos_Bmin30_Bmax200_thr0_massOver2_v10'
output_format = 'csv'

## Process data

In [4]:
# Loop over observed histograms and rebin signal
def process_data(bsm_signal):
    # Identify observed histograms
    root_file = f'{observed_dir}/{bsm_signal}/{file_name}.root'
    tfile = ROOT.TFile.Open(root_file ,"READ")
    hists_names = [k.GetName() for k in tfile.GetListOfKeys()]
    tfile.Close()

    # Process histograms
    hists_data = {'X' : [], 'M' : [], 'C' : [], 'S' : [], 'B' : []}
    not_found = []

    print(f'> Found {len(hists_names)} histograms...')
    for hist_name in tqdm(hists_names):

        if 'hCat' not in hist_name: continue

        # Get observed histograms and binning with mass values
        tfile = ROOT.TFile.Open(root_file ,"READ")
        th1 = tfile.Get(hist_name)
        if not th1: print(f"Failed to get histogram\n hist_name = {hist_name}\n root_file = {root_file}")
        th1.SetDirectory(0)
        tfile.Close()

        entries = np.array([th1.GetBinContent(b) for b in range(1,th1.GetXaxis().GetNbins()+1)])
        bin_edges = np.array(list(th1.GetXaxis().GetXbins()))
        bin_sizes = np.array([bin_edges[i+1]-bin_edges[i] for i in range(0, len(bin_edges)-1)])
        bin_centers = bin_edges[:-1] + bin_sizes/2

        hists_data['X'] += [entries]
        hists_data['M'] += [bin_edges]
        hists_data['C'] += [np.array([hist_name])]

        # Get signal raw histogram
        cat, obs = hist_name.split('__')
        sig_cat = cat.replace('hCat', 'Cat')
        if hist_name.find('OS')!=-1: 
            sig_hist = f'{cat.split("OS_")[0].replace("hCat", "hrawCat")+"OS"}__{obs}'
        elif hist_name.find('SS')!=-1:
            sig_hist = f'{cat.split("SS_")[0].replace("hCat", "hrawCat")+"SS"}__{obs}'
        else:
            sig_hist = f'{cat.split("m_")[0].replace("hCat", "hrawCat")+"m"}__{obs}'

        sig_root_file = f'{signal_dir}/{bsm_signal}/histo_raw/{sig_cat}.root'
        
        if not Path(sig_root_file).is_file():
            # print(f'Could not find signal histogram for {hist_name}. Skipping...')
            not_found += [hist_name]
            hists_data['S'] += [np.ones(entries.shape[0])]
            hists_data['B'] += [entries]
            continue
        
        tfile = ROOT.TFile.Open(sig_root_file ,"READ")
        th1 = tfile.Get(sig_hist)
        if not th1: print("Failed to get histogram\n hist_name = %s\n root_file = %s" % (sig_hist, sig_root_file)); continue
        th1.SetDirectory(0)
        tfile.Close()

        raw_entries = np.array([th1.GetBinContent(b) for b in range(1,th1.GetXaxis().GetNbins()+1)])
        raw_bin_edges = np.array(list(th1.GetXaxis().GetXbins()))
        raw_bin_widths = np.array([raw_bin_edges[i+1]-raw_bin_edges[i] for i in range(0, len(raw_bin_edges)-1)])
        raw_bin_centers = raw_bin_edges[:-1] + raw_bin_widths/2

        # Rebin signal
        hist = rebin(th1, bin_edges)

        sig_entries = np.array([hist.GetBinContent(b) for b in range(1,hist.GetXaxis().GetNbins()+1)])
        bkg_entries = entries - sig_entries
        # bin_edges = np.array(list(hist.GetXaxis().GetXbins()))
        # bin_widths = np.array([bin_edges[i+1]-bin_edges[i] for i in range(0, len(bin_edges)-1)])
        # bin_centers = bin_edges[:-1] + bin_widths/2

        hists_data['S'] += [sig_entries]
        hists_data['B'] += [bkg_entries]

    # Print statistics
    nbins = np.array([len(h) for h in hists_data['X']])
    ymin = np.min(np.array([np.min(h) for h in hists_data['X']]))
    ymax = np.max(np.array([np.max(h) for h in hists_data['X']]))
    xmin = np.min(np.array([np.min(h) for h in hists_data['M']]))
    xmax = np.max(np.array([np.max(h) for h in hists_data['M']]))
    print(f'> Total of {len(hists_data["X"])} histograms with\n'
          f'  {np.min(nbins)} <= Nbins <= {np.max(nbins)}\n'
          f'  {ymin} <= entries per bin <= {ymax}\n'
          f'  {xmin} <= mass <= {xmax}'
          )
    print(f'Signal not found for {len(not_found)} histograms')

    # Save histograms into DDP input format
    for tag, data in hists_data.items():
        if output_format == 'npy':
            # Create numpy arrays
            out_file = f'{output_dir}/{bsm_signal}/{tag}.npy'
            np.save(out_file, np.array(data, dtype=object))
        elif output_format == 'csv':
            df_out = pd.DataFrame(data=data)
            out_file = f'{output_dir}/{bsm_signal}/{tag}.csv'
            df_out.to_csv(out_file, index=False, header=False)
        else:
            print(f'ERROR: current supported output formats are npy and csv. "{output_format}" is not supported')
        print(f'> Saved histograms to {out_file}')
    
    

In [5]:
# channel 3
bsm_signals = [
# 'LQ_blbl_test_1300_chan3_3000',
# 'LQ_tlbv_test_1300_chan3_1000',
# 'LQ_tvtv_test_1300_chan3_1000',
# 'Wprime_lvqq_HVT_chan3_10000',
# 'Wprime_lvqq_HVT_chan3_100000',
# 'Wprime_qqvv_HVT_chan3_10000',
# 'Wprime_qqvv_HVT_chan3_100000',
# 'hh_tbtb_chan3_10000',
# 'hh_tbtb_chan3_100000',
# 'sqsq1_sq1400_neut800_chan3',
# '2HDM_GGF_chan3_10000',
# '2HDM_GGF_chan3_100000',
'stlp_st1000_chan3'
]

# channel 2a
# bsm_signals = [
#     'LQ_bebe_600_chan2a_100',
#     'LQ_bebe_600_chan2a_1000',
#     'LQ_bebmu_600_chan2a_10',
#     'LQ_bebmu_600_chan2a_100',
#     'LQ_bebmu_600_chan2a_1000',
#     'LQ_bmubmu_600_chan2a_100',
#     'LQ_bmubmu_600_chan2a_1000',
#     'pp23mt_50_chan2a',
#     'pp24mt_50_chan2a'
# ]

# channel 2b
# bsm_signals = [
# 'stlp_st1000_chan2b',
#     'LQ_bebe_600_chan2b_100',
#     'LQ_bebe_600_chan2b_1000',
#     'LQ_bebmu_600_chan2b_10',
#     'LQ_bebmu_600_chan2b_100',
#     'LQ_bebmu_600_chan2b_1000',
#     'LQ_bmubmu_600_chan2b_100',
#     'LQ_bmubmu_600_chan2b_1000',
#     'pp23mt_50_chan2b',
#     'pp24mt_50_chan2b'
#     ]

# channel 1
# bsm_signals = [
# 'stlp_st1000_chan1',
# '2HDM_GGF_chan1_10000',
#     '2HDM_GGF_chan1_100000',
#     'LQ_bebe_600_chan1_100',
#     'LQ_bebe_600_chan1_1000',
#     'LQ_bebmu_600_chan1_10',
#     'LQ_bebmu_600_chan1_100',
#     'LQ_bebmu_600_chan1_1000',
#     'LQ_bmubmu_600_chan1_100',
#     'LQ_bmubmu_600_chan1_1000',
#     'Wprime_lvqq_HVT_chan1_10000',
#     'Wprime_lvqq_HVT_chan1_100000',
#     'Wprime_qqvv_HVT_chan1_10000',
#     'Wprime_qqvv_HVT_chan1_100000',
#     'hh_tbtb_chan1_10000',
#     'hh_tbtb_chan1_100000'
# ]

In [6]:
for bsm_signal in bsm_signals:
    print(f'> Processing {bsm_signal}...')
    Path(f'{output_dir}/{bsm_signal}').mkdir(parents=True, exist_ok=True)
    process_data(bsm_signal)

> Processing stlp_st1000_chan3...
> Found 18281 histograms...


100%|██████████| 18281/18281 [26:55<00:00, 11.32it/s]


> Total of 18281 histograms with
  31 <= Nbins <= 118
  0.0 <= entries per bin <= 184802.0
  48.0 <= mass <= 9481.0
Signal not found for 996 histograms
> Saved histograms to /project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3/stlp_st1000_chan3/X.csv
> Saved histograms to /project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3/stlp_st1000_chan3/M.csv
> Saved histograms to /project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3/stlp_st1000_chan3/C.csv
> Saved histograms to /project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3/stlp_st1000_chan3/S.csv
> Saved histograms to /project/def-arguinj/shared/DDP_data/2024-01-25_dark_machines_analysis12_rebinned_bsm_signals/v10_3/stlp_st1000_chan3/B.csv


Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_0g_1e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_0g_1e_0m: mass(j0,j2) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_0g_1e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_0g_1e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_1g_0e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_1g_0e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_1g_0e_0m: mass(j0,j1) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_0Wh_0T_0HM_0Z_1g_1e_0m: mass(j1,j2) (Potential memory leak).
Warning in <TROOT::Append>: Replacing existing TH1: Cat_1Wh_0T_0